<a href="https://colab.research.google.com/github/manisharan-deep/Explainable-AI-Lab-Assignment/blob/main/Explainable_AI_Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1 Assignment Problem 1: Car Evaluation Prediction
 Problem Statement:
 Car Evaluation dataset classifies car quality. LIME shows decisive features.
 Tasks:
1. Load dataset
2. Train Decision Tree
3. Apply LIME
4. Interpret contributions
 Deliverables:
 Code
 Outputs
 Report

In [13]:
# ==============================
# Assignment Problem 1:
# Car Evaluation Prediction with LIME-style Explanations
# ==============================

import os
import io
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import Ridge

# ---------- 1) Load dataset ----------
csv_path = "car_evaluation.csv"   # <-- put your dataset path here
df = pd.read_csv(csv_path)

# Expected columns for Car Evaluation dataset
expected_cols = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
if set(expected_cols).issubset(df.columns):
    pass
else:
    if df.shape[1] == 7:
        df.columns = expected_cols

target_col = "class"
feature_cols = [c for c in df.columns if c != target_col]

X = df[feature_cols].astype(str)
y = df[target_col].astype(str)

# ---------- 2) Train Decision Tree ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols)]
)

clf = DecisionTreeClassifier(max_depth=5, min_samples_split=4, random_state=42)

pipe = Pipeline(steps=[("prep", preprocess), ("model", clf)])
pipe.fit(X_train, y_train)

# ---------- 3) Evaluation ----------
y_pred = pipe.predict(X_test)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=pipe.classes_)
report = classification_report(y_test, y_pred)

print("Accuracy:", acc)
print("\nClassification Report:\n", report)

# ---------- 4) Global Feature Importance ----------
ohe: OneHotEncoder = pipe.named_steps["prep"].named_transformers_["cat"]
ohe_feature_names = ohe.get_feature_names_out(feature_cols)

tree_importances = pipe.named_steps["model"].feature_importances_
imp_df = pd.DataFrame({"Feature": ohe_feature_names, "Importance": tree_importances})
print("\nTop 10 important features:\n", imp_df.sort_values("Importance", ascending=False).head(10))

# ---------- 5) LIME-style Local Explanation ----------
rng = np.random.RandomState(0)
cat_values = {c: X_train[c].value_counts(normalize=True) for c in feature_cols}

def sample_perturbation(x0_row, p_change=0.3):
    x_new = {}
    for c in feature_cols:
        if rng.rand() < p_change:
            vals = cat_values[c].index.values
            probs = cat_values[c].values
            x_new[c] = rng.choice(vals, p=probs)
        else:
            x_new[c] = x0_row[c]
    return x_new

def hamming_distance_row(xa, xb):
    mismatches = sum(1 for c in feature_cols if xa[c] != xb[c])
    return mismatches / len(feature_cols)

def explain_instance(x0_row, pipeline, n_samples=1000, kernel_width=0.75):
    x0_df = pd.DataFrame([x0_row], columns=feature_cols)
    pred_class = pipeline.predict(x0_df)[0]

    samples = [sample_perturbation(dict(zip(feature_cols, x0_row))) for _ in range(n_samples)]
    Xp = pd.DataFrame(samples)

    dists = np.array([hamming_distance_row(s, dict(zip(feature_cols, x0_row))) for _, s in Xp.iterrows()])
    weights = np.exp(-(dists**2) / (kernel_width**2))

    Z = pipeline.named_steps["prep"].transform(Xp)
    z0 = pipeline.named_steps["prep"].transform(x0_df)[0].toarray().flatten() # Convert sparse to dense and flatten

    class_index = list(pipeline.named_steps["model"].classes_).index(pred_class)
    probs = pipeline.predict_proba(Xp)[:, class_index]

    model = Ridge(alpha=1.0, fit_intercept=True)
    model.fit(Z, probs, sample_weight=weights)

    coefs = model.coef_
    active_idx = np.where(z0 == 1)[0]
    contribs = [(ohe_feature_names[i], coefs[i]) for i in active_idx]
    return pred_class, contribs

# Example: explain one test instance
idx = X_test.index[0]
x0 = X_test.loc[idx]
true_label = y_test.loc[idx]

pred_class, contributions = explain_instance(x0.values, pipe)
print(f"\nTrue label: {true_label}, Predicted: {pred_class}")
print("Local explanation (feature contributions):")
for f, w in contributions:
    print(f"  {f}: {w:.4f}")

Accuracy: 0.8796296296296297

Classification Report:
               precision    recall  f1-score   support

         acc       0.69      0.94      0.79        96
        good       0.52      0.71      0.60        17
       unacc       1.00      0.92      0.96       303
       vgood       0.00      0.00      0.00        16

    accuracy                           0.88       432
   macro avg       0.55      0.64      0.59       432
weighted avg       0.87      0.88      0.87       432


Top 10 important features:
          Feature  Importance
19    safety_low    0.377648
12     persons_2    0.255496
5      maint_low    0.125944
7    maint_vhigh    0.080958
3   buying_vhigh    0.064538
0    buying_high    0.044259
6      maint_med    0.044037
18   safety_high    0.007121
1     buying_low    0.000000
4     maint_high    0.000000

True label: unacc, Predicted: unacc
Local explanation (feature contributions):
  buying_low: -0.0155
  maint_med: -0.0029
  doors_3: 0.0013
  persons_2: 0.5041
  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


 Problem Statement:
 Mushroom dataset predicts edible vs poisonous. LIME explains classification.
 Tasks:
1. Load dataset
2. Train Random Forest
3. Apply LIME
4. Interpret results
 Deliverables:
 Code
 Outputs
 Report

In [17]:
# ==============================
# Assignment Problem 2:
# Mushroom Classification with LIME Explanations
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import lime
import lime.lime_tabular

# ---------- 1) Load dataset ----------
df = pd.read_csv("/content/secondary_data.csv.zip", sep=';')   # <-- change path if needed
print("Dataset shape:", df.shape)
print(df.head())

# Assume target column is named 'class' (edible/poisonous)
target_col = "class"
X = df.drop(columns=[target_col])
y = df[target_col]

# ---------- 2) Train Random Forest ----------
categorical_features = list(X.columns)
preprocess = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)]
)

rf = RandomForestClassifier(n_estimators=200, random_state=42)

pipe = Pipeline(steps=[("prep", preprocess), ("model", rf)])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
pipe.fit(X_train, y_train)

# ---------- 3) Evaluation ----------
y_pred = pipe.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# ---------- 4) Apply LIME ----------
# Fit LIME explainer on training data
ohe = pipe.named_steps["prep"].named_transformers_["cat"]
feature_names = ohe.get_feature_names_out(categorical_features)

explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=pipe.named_steps["prep"].transform(X_train).toarray(),
    feature_names=feature_names,
    class_names=pipe.named_steps["model"].classes_,
    mode="classification"
)

# Pick one instance from test set
i = 0
x0 = X_test.iloc[i]
print("\nExplaining instance:", x0.to_dict())

exp = explainer.explain_instance(
    pipe.named_steps["prep"].transform(x0.to_frame().T).toarray()[0],
    pipe.named_steps["model"].predict_proba,
    num_features=10
)

# Show explanation in console
print("\nLIME Explanation:\n", exp.as_list())

# Save visualization
exp.save_to_file("lime_mushroom_example.html")
print("\nLIME explanation saved as lime_mushroom_example.html")

Dataset shape: (61069, 21)
  class  cap-diameter cap-shape cap-surface cap-color does-bruise-or-bleed  \
0     p         15.26         x           g         o                    f   
1     p         16.60         x           g         o                    f   
2     p         14.07         x           g         o                    f   
3     p         14.17         f           h         e                    f   
4     p         14.64         x           h         o                    f   

  gill-attachment gill-spacing gill-color  stem-height  ...  stem-root  \
0               e          NaN          w        16.95  ...          s   
1               e          NaN          w        17.99  ...          s   
2               e          NaN          w        17.80  ...          s   
3               e          NaN          w        15.77  ...          s   
4               e          NaN          w        16.53  ...          s   

  stem-surface stem-color veil-type veil-color has-ring rin